In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. Add uv to the current shell path
import os
os.environ["PATH"] += ":/root/.cargo/bin"

# 3. Create/Use a venv and install
!uv venv .venv
!source .venv/bin/activate
!uv pip install torch transformers accelerate transformer_lens huggingface_hub

downloading uv 0.11.28 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Using Python 3.12.13 environment at: /usr
Resolved 97 packages in 1.37s                                        
Prepared 6 packages in 503ms                                             
Installed 6 packages in 16ms.1                              
 + better-abc==0.0.3
 + fancy-einsum==0.0.3
 + jaxtyping==0.3.11
 + transformer-lens==3.5.1
 + transformers-stream-generator==0.0.5
 + wadler-lindig==0.1.7


In [2]:
from getpass import getpass
hf_token = getpass("Hugging Face token: ")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import os
# import torch
# import json
# from huggingface_hub import hf_hub_download
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # 1. Download the specific JSONL file directly (Bypasses 'datasets' lib)
# print("Downloading sycophancy eval data...")
# file_path = hf_hub_download(
#     repo_id="Anthropic/model-written-evals", 
#     filename="sycophancy/sycophancy_on_nlp_survey.jsonl",  # Corrected filename
#     repo_type="dataset"
# )

# # 2. Load manually
# with open(file_path, 'r') as f:
#     eval_data = [json.loads(line) for line in f]
# print(f"Loaded {len(eval_data)} adversarial prompts.")

# # 3. Setup Model (Ensure hf_token is defined)
# model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m", token=hf_token).to("cuda")
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m", token=hf_token)

# # 4. Collection Loop
# save_dir = "/content/drive/MyDrive/gemma270M-SAE"
# os.makedirs(save_dir, exist_ok=True)
# all_activations = []
# storage = {}

# def hook_fn(module, input, output):
#     storage['activation'] = output.detach()

# # Register on Layer 8 MLP
# hook_handle = model.model.laydoers[8].mlp.register_forward_hook(hook_fn)

# print("Starting collection...")
# for i, example in enumerate(eval_data):
#     # This dataset uses 'question' as the field name
#     text = example.get('question', '')
#     if len(text) < 20: continue

#     inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to("cuda")
#     with torch.no_grad():
#         _ = model(**inputs)
    
#     all_activations.append(storage['activation'].squeeze(0))
#     if i % 100 == 0: 
#         print(f"Collected {len(all_activations)} prompts...")
#     if len(all_activations) >= 3000: 
#         break # Collect up to 3000 sycophancy examples

# # 5. Save
# dataset_activations = torch.cat(all_activations, dim=0)
# torch.save(dataset_activations, os.path.join(save_dir, "gemma_layer8_acts.pt"))
# hook_handle.remove()
# print(f"Saved {dataset_activations.shape} activations to Drive.")

In [10]:
# Building the SAE
import os 
import torch

save_dir = "/content/drive/MyDrive/gemma270M-SAE"

# Load the activations
dataset_activations = torch.load(os.path.join(save_dir, "gemma_layer8_acts.pt"))

# Define the SAE architecture
print(f"Data ready: {dataset_activations.shape}")

Data ready: torch.Size([384000, 640])


In [6]:
from locale import nl_langinfo
import torch
import torch.nn as nn
import torch.nn.functional  as F

# Define the SAE architecture
class SAE(nn.Module):
    def __init__(self, n_input, n_latent):
        super(SAE, self).__init__()
        self.encoder = nn.Linear(n_input, n_latent) # 640->4096
        self.decoder = nn.Linear(n_latent, n_input, bias=False) # 4096->640

        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)

    def forward(self, x):
        # Encode
        latent = F.relu(self.encoder(x))

        # Decode

        reconsstruction = self.decoder(latent)
        return latent , reconsstruction

In [ ]:
import gc

if 'model' in globals(): del model
if 'tokenizer' in globals(): del tokenizer

gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared!")

In [5]:
# Training SAE

n_input = 640
n_latent = 4096

sae = SAE(n_input , n_latent).to("cuda" if torch.cuda.is_available() else "cpu")
data = dataset_activations.to(torch.float32).to(sae.encoder.weight.device)

optimizer = torch.optim.Adam(sae.parameters(), lr = 1e-3)
batch_size = 4096
epochs = 20 

print("Training starts...")
for stage in ["warmup","sparsity"]:
    l1_coeff = 1e-4 if stage =="sparsity" else 0.0
    print(f"\n --- Stage: {stage.upper()} (L1 Coeff = {l1_coeff})---")

    for epoch in range(epochs):
        #Shuffle the indices of input for better training
        permute = torch.randperm(data.size(0))
        epoch_recon_loss = 0.0
        epoch_sparsity_loss = 0.0

        # Batch Loop
        for i in range(0, data.size(0),batch_size):
            indices = permute[i:i+ batch_size]
            batch_data = data[indices]

            # Forward
            latent, reconstruction = sae(batch_data)
            reconstruction_loss = F.mse_loss(reconstruction,batch_data)
            sparsity_loss = l1_coeff * latent.abs().sum(dim=-1).mean()
            loss = reconstruction_loss + sparsity_loss

            #backprop
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_recon_loss += reconstruction_loss.item()
            epoch_sparsity_loss += sparsity_loss.item()

        #Log average loss across batches
        num_batches = data.size(0) //batch_size
        print(f"Epoch {epoch+1}/{epochs}: Recon Loss = {epoch_recon_loss/num_batches:.6f} | Sparsity = {epoch_sparsity_loss/num_batches:.6f}")


NameError: name 'SAE' is not defined

# Common Dim error

Make sure if model is BF16 , Bf16 -> float32 is fine , fp32 to bf16 can cause issues

# Looking at the features

In [11]:
import torch

# 1. Clear cache
torch.cuda.empty_cache()

batch_size = 4096
n_tokens = dataset_activations.size(0)
n_latent = 4096

# We will accumulate feature sums batch-by-batch (takes virtually 0 memory)
feature_sums = torch.zeros(n_latent, dtype=torch.float32)

print("Streaming evaluation (0% memory overhead)...")
with torch.no_grad():
    for i in range(0, n_tokens, batch_size):
        batch = dataset_activations[i:i+batch_size].to(torch.float32).to("cuda")
        latent_batch, _ = sae(batch)
        
        # Accumulate the sum of activations for each feature
        feature_sums += latent_batch.cpu().sum(dim=0)

# Calculate statistics
feature_means = feature_sums / n_tokens
alive_mask = (feature_sums > 0.001)

print(f"\nNumber of 'Alive' features: {alive_mask.sum().item()} out of {n_latent}")

# Get top 5 most active features
top_means = torch.topk(feature_means, k=5)
print("Top 5 Feature Indices:", top_means.indices.tolist())

# 2. Get top activations specifically for the #1 active feature
# Instead of storing 4096 features, we only store a single [384,000] array (1.5 Megabytes!)
feat_idx = top_means.indices[0].item()
feature_activation = torch.zeros(n_tokens, dtype=torch.float32)

print(f"\nRunning target analysis on Feature {feat_idx}...")
with torch.no_grad():
    for i in range(0, n_tokens, batch_size):
        batch = dataset_activations[i:i+batch_size].to(torch.float32).to("cuda")
        latent_batch, _ = sae(batch)
        feature_activation[i:i+batch_size] = latent_batch[:, feat_idx].cpu()

# Find the top 10 spikes
top_val, top_idx = torch.topk(feature_activation, k=10)

print(f"\n--- Analysis for Feature {feat_idx} ---")
print(f"Top 5 peak values: {top_val[:5].tolist()}")

print("\nTop Spikes by token position:")
for val, idx in zip(top_val, top_idx):
    print(f"  Value: {val:.2f} | Token position: {idx.item()}")

Streaming evaluation (0% memory overhead)...

Number of 'Alive' features: 3638 out of 4096
Top 5 Feature Indices: [2281, 2653, 1934, 402, 1450]

Running target analysis on Feature 2281...

--- Analysis for Feature 2281 ---
Top 5 peak values: [0.32130590081214905, 0.32004183530807495, 0.31818971037864685, 0.31726324558258057, 0.3157496452331543]

Top Spikes by token position:
  Value: 0.32 | Token position: 163455
  Value: 0.32 | Token position: 280959
  Value: 0.32 | Token position: 168828
  Value: 0.32 | Token position: 175998
  Value: 0.32 | Token position: 177650
  Value: 0.32 | Token position: 169471
  Value: 0.31 | Token position: 291069
  Value: 0.31 | Token position: 69112
  Value: 0.31 | Token position: 167290
  Value: 0.31 | Token position: 232562


In [ ]:
weights_path = "/content/drive/MyDrive/gemma270M-SAE/sae_weights.pt"
torch.save(sae.state_dict(), weights_path)
print(f"SAE weights saved successfully to {weights_path}!")

In [7]:
# To load later:
sae = SAE(640, 4096).to("cuda" if torch.cuda.is_available() else "cpu")
sae.load_state_dict(torch.load("/content/drive/MyDrive/gemma270M-SAE/sae_weights.pt"))
sae.eval() # Set to evaluation mode

SAE(
  (encoder): Linear(in_features=640, out_features=4096, bias=True)
  (decoder): Linear(in_features=4096, out_features=640, bias=False)
)

In [12]:
# 1. Identify which prompts trigger Feature 2281
feat_idx = 2281 
# We need to get the feature activations again
latent, _ = sae(dataset_activations.to(torch.float32).to(sae.encoder.weight.device))

# 2. Get top tokens for this specific feature
top_vals, top_indices = torch.topk(latent[:, feat_idx], k=10)

# 3. Map back to text
# This is a bit manual, but we'll print the prompt index and the token
print(f"Analyzing Feature {feat_idx}...")
for val, idx in zip(top_vals, top_indices):
    # This assumes your dataset_activations were stacked in order
    print(f"Value: {val:.3f} | Global Token Index: {idx.item()}")

Analyzing Feature 2281...
Value: 0.321 | Global Token Index: 163455
Value: 0.320 | Global Token Index: 280959
Value: 0.318 | Global Token Index: 168828
Value: 0.317 | Global Token Index: 175998
Value: 0.316 | Global Token Index: 177650
Value: 0.315 | Global Token Index: 169471
Value: 0.315 | Global Token Index: 291069
Value: 0.314 | Global Token Index: 69112
Value: 0.314 | Global Token Index: 167290
Value: 0.313 | Global Token Index: 232562
